In [ ]:
# -*- coding: utf-8 -*-
"""
"""

import os
import numpy as np
import rasterio
from datetime import datetime
from tqdm import tqdm
import glob

# ===================== Parameter settings =====================
data_dir = r"...\daily"
output_file = r"...\TNin90.tif"

start_year, end_year = 1981, 2010
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# ===================== Build daily file index =====================
# Get all tif file paths
all_files = sorted(glob.glob(os.path.join(data_dir, "*.tif")))

# Filter files within the reference period
def valid_date(fn):
    try:
        date_str = os.path.basename(fn)[3:11]  # Extract YYYYMMDD
        date = datetime.strptime(date_str, "%Y%m%d")
        return start_year <= date.year <= end_year and not (date.month == 2 and date.day == 29)
    except:
        return False

all_files = [f for f in all_files if valid_date(f)]
print(f"Found {len(all_files)} valid daily files")

# ===================== Read the first file to get metadata =====================
with rasterio.open(all_files[0]) as src:
    meta = src.meta.copy()
    height, width = src.height, src.width
    transform, crs = src.transform, src.crs
    nodata = src.nodata

# Initialize result array: 365 days × height × width
tnin90 = np.full((365, height, width), np.nan, dtype=np.float32)

# Initialize container grouped by day-of-year
all_data = {d: [] for d in range(365)}

# ===================== Read files and organize by day-of-year =====================
for file in tqdm(all_files, desc="Reading daily DTR"):
    date_str = os.path.basename(file)[3:11]
    date = datetime.strptime(date_str, "%Y%m%d")
    day_of_year = (date - datetime(date.year, 1, 1)).days  # 0-based (0~364)
    if day_of_year >= 365:
        continue  # Skip February 29

    with rasterio.open(file) as src:
        data = src.read(1).astype(np.float32)
        if nodata is not None:
            data[data == nodata] = np.nan
        all_data[day_of_year].append(data)

# ===================== Compute the 90th percentile =====================
for day in tqdm(range(365), desc="Computing TNin90"):
    window_data = []
    for offset in range(-7, 8):
        day_idx = (day + offset) % 365  # Circular window
        window_data.extend(all_data[day_idx])
    if len(window_data) == 0:
        continue
    window_stack = np.stack(window_data, axis=0)
    tnin90[day] = np.nanpercentile(window_stack, 90, axis=0)

# ===================== Save output =====================
meta.update({
    "count": 365,
    "dtype": "float32",
    "compress": "lzw",
    "nodata": -9999
})

with rasterio.open(output_file, "w", **meta) as dst:
    for day in range(365):
        out_data = np.where(np.isnan(tnin90[day]), -9999, tnin90[day])
        dst.write(out_data, day + 1)
        dst.set_band_description(day + 1, f"Day-{day + 1}")

print("✅ TNin90 calculation completed.")
print("Output saved to:", output_file)